# N115 · 博弈DP与得分差

**目标：** 在双方最优前提下使用当前玩家视角。

**先修：** N098, N107, N062。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** minimax；零和；先后手；得分差；缓存；与概率决策区别。

**配套讲解来源：** [同名逐章意见](../../comment/115_game_dp_minimax.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章对付的是"两个绝顶聪明的人轮流拿分"的游戏题。你不能再像普通 DP 那样只考虑自己怎么拿最多——因为你拿完之后，对手也会用同样的聪明程度拿他的最多。本章三个具体游戏：

1. **两端取数（Predict the Winner）：** 一排数字，两人轮流，每人每次只能拿走最左边或最右边那个数，问先手最终总分是否不低于后手。
2. **Stone Game II：** 一排非负的石堆，从左往右拿；轮到你时可以拿走最前面的 1 到 2M 堆（M 初始为 1），你拿了 x 堆之后 M 更新为 max(M,x)；两人都拿最优，问先手最多拿到多少颗石子。
3. **Stone Game III：** 一排石子（可能有负数！），从左往右每次拿 1 到 3 颗，问最后谁分高（输出 Alice / Bob / Tie）。

**具体到数字的例子：** 输入 `[1,5,2]`（两端取数），输出是"先手赢不了"。为什么？先手拿 1，剩下 [5,2] 轮到对手，对手必然拿 5，先手再拿 2，比分 3:5；先手拿 2，对手面对 [1,5] 必然拿 5，比分 3:5——怎么走都输。再比如 Stone Game III 输入 `[1,2,3,7]`：我们会手算出先手最优也只能力保 −1 分的差距，所以答案是 Bob 赢。

## 2. 基础知识：先读懂这些词

- **博弈DP：** 状态里记录"轮到谁走、这个局面下轮到的人最多能领先多少"，然后按游戏规则转移。它和普通 DP 的唯一区别是：每走一步，"当前行动者"就换人，你的收益和对手的收益绑在一起。
- **零和（zero-sum）：** 总分是固定的，你多拿一颗，对手就少拿一颗。因为总和不变，我们不需要分别记两个人的分数，只记"差值"就够描述整个局面。
- **当前玩家视角：** 所有子问题统一成"现在轮到的人，从这个局面出发最终最多领先多少分"。这样方程两边都是同一个含义的数，不需要给两个玩家各开一维。
- **得分差（difference）：** 上面说的"领先多少"。核心递推是"我的差 = 我这一步的收获 − 我走完之后对手的差"。对手的差要被减掉，因为下一个状态的优势属于对手。
- **minimax（极大极小）：** 我每一步取所有选项中对我最好的（max）；由于零和，"对我最好"自动就是"对对手最坏"（相当于替对手做了 min）。所以一个 max 加一个减号，就实现了双方轮流最优。
- **缓存（`@cache`）：** 把 (局面) 到 (得分差) 的答案存起来，重复局面直接查表。notebook 里 Stone Game II 用了它，因为它的状态是 (位置, M) 两个参数。
- **后缀和（suffix sum）：** suffix[i] = 从第 i 堆到最后一堆的总石子数。预先算好，配合"我的所得 = 剩余总量 − 对手所得"使用，就不用在转移里反复求和。
- **与概率决策的区别：** 概率 DP（上一章）的对手是" randomness"，你要按概率加权平均（求期望）；博弈 DP 的对手是"聪明人"，你要假设他走对你最坏的一步（求最差情况的最大值）。一个是平均，一个是最坏。

## 3. 思路推导：从小例子开始

### 3.1 两端取数：区间上的得分差

- **Step 1：** 定义 d[l][r] = "面对 nums[l..r] 的当前玩家，最优走下去能领先多少分"。
- **Step 2：** 只有两种选择：拿左端 nums[l]，然后对手面对 [l+1,r]；或拿右端 nums[r]，然后对手面对 [l,r−1]。所以 d[l][r] = max(nums[l] − d[l+1][r], nums[r] − d[l][r−1])。减号就是把"对手的领先"翻译成"我的落后"。
- **Step 3：** 区间长度为 1 时，d = 那个数字本身；空区间差为 0。
- **Step 4：** 答案是 d[0][n−1] ≥ 0（先手不落后就算赢，平局算先手赢）。

**手算 `[1,5,2]`：** 长度 1：d[0,0]=1，d[1,1]=5，d[2,2]=2。长度 2：d[0,1]=max(1−5, 5−1)=4（区间 [1,5]，当前玩家拿 5 领先 4）；d[1,2]=max(5−2, 2−5)=3。长度 3：d[0,2]=max(1−d[1,2], 2−d[0,1])=max(1−3, 2−4)=−2。−2 < 0，先手必输，返回 False，和函数结果一致。

### 3.2 Stone Game II：把 M 放进状态

- **Step 1：** 状态是 (i, M)：还剩 piles[i:]，当前上限 M。f(i,M) = 当前玩家从这儿出发最多拿多少颗。
- **Step 2：** 如果 i+2M ≥ n，剩下的堆数不超过你能拿的上限，直接全拿（石子都非负，多拿不会亏）。
- **Step 3：** 否则枚举拿 x = 1..2M 堆：你拿完后对手面对 (i+x, max(M,x))，他能拿 f(i+x, max(M,x))；由于剩余总量固定是 suffix[i]，你的所得 = suffix[i] − 对手所得。取所有 x 里最大的。

**手算 `[2,7,9,4,4]`（总 26，suffix = [26,24,17,8,4]）：** 关键几步：f(3,1)、f(4,2) 都满足"能全拿"，分别是 8 和 4。f(2,1)：可拿 1 堆得 17−f(3,1)=9，或拿 2 堆得 17−f(4,2)=13，取 13。f(3,2)：3+2×2≥5，全拿 8。f(1,1)：拿 1 堆得 24−f(2,1)=11，拿 2 堆得 24−f(3,2)=16，取 16。最后 f(0,1)：拿 1 堆得 26−f(1,1)=10，拿 2 堆得 26−f(2,2)=26−17=9，取 **10**——与断言 `stone_game_ii([2,7,9,4,4])==10` 一致。实际对局是：Alice 拿 1 堆（2），Bob 拿 2 堆（7+9），M 变 2，Alice 把剩下两堆 4+4 全拿走，Alice 共 2+8=10。

### 3.3 Stone Game III：允许负数的取前缀

- **Step 1：** dp[i] = 面对 stone_value[i:] 的当前玩家的最优得分差。
- **Step 2：** 枚举拿 j−i+1 = 1..3 颗：拿掉 stone_value[i..j] 这一段（和记作 taken），然后对手面对 [j+1:]，dp[i] = max(taken − dp[j+1])。
- **Step 3：** 因为石子可能是负数，就算三个选项全亏也要挑亏得最少的，所以初始值用负无穷。

**手算 `[1,2,3,7]`（“运行示例”部分的表格就是这个过程）：** 从右往左：
- i=3：只能拿 7，选项 [7−0]=[7]，dp[3]=7；
- i=2：拿 1 颗得 3−7=−4；拿 2 颗得 (3+7)−0=10，dp[2]=10；
- i=1：拿 1 颗 2−10=−8；拿 2 颗 5−7=−2；拿 3 颗 12−0=12，dp[1]=12；
- i=0：拿 1 颗 1−12=−11；拿 2 颗 3−10=−7；拿 3 颗 6−7=−1，dp[0]=−1。
dp[0]=−1 < 0，Bob 赢。表格的"取1..3项得分差"列就是每行的选项列表 [−11,−7,−1] 等，你可以逐格核对。

## 4. 核心代码：predict_the_winner

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def predict_the_winner(nums):
    if not nums:
        return True
    dp = list(nums)
    for length in range(2, len(nums) + 1):
        for l in range(len(nums) - length + 1):
            r = l + length - 1
            dp[l] = max(nums[l] - dp[l + 1], nums[r] - dp[l])
    return dp[0] >= 0
```

### 逐段读懂代码

### 4.1 `predict_the_winner(nums)`

```python
def predict_the_winner(nums):
    if not nums:
        return True
    dp = list(nums)
    for length in range(2, len(nums) + 1):
        for l in range(len(nums) - length + 1):
            r = l + length - 1
            dp[l] = max(nums[l] - dp[l + 1], nums[r] - dp[l])
    return dp[0] >= 0
```

- `if not nums: return True`：空输入时谁也拿不到分，差为 0，按"先手不输即赢"返回 True，同时挡掉空数组。
- `dp=list(nums)`：长度为 1 的区间，当前玩家只能拿走唯一的数，差就是这个数。注意 `list(nums)` 是拷贝，不改动原数组。
- 两层循环按区间长度从小到大：`r=l+length-1` 算出右端点。
- `dp[l]=max(nums[l]-dp[l+1],nums[r]-dp[l])` 是全函数最精妙的一行，只用一维数组干了二维的事：在计算区间 [l,r] 时，`dp[l+1]` 在本轮内层循环里已经被更新成区间 [l+1,r]（长度 length−1）的答案；而 `dp[l]` 还保留着上一轮的值，恰好是区间 [l,r−1]（也是长度 length−1）的答案。于是"拿左端"用 `nums[l]-dp[l+1]`（我拿 nums[l]，减去对手在 [l+1,r] 的领先），"拿右端"用 `nums[r]-dp[l]`（我拿 nums[r]，减去对手在 [l,r−1] 的领先）。
- `return dp[0]>=0`：循环结束时 dp[0] 是整个数组的差；≥0 表示先手不落后（平局也算先手赢），这是题目的约定。

### 4.2 `stone_game_ii(piles)`

```python
def stone_game_ii(piles):
    n = len(piles)
    suffix = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suffix[i] = suffix[i + 1] + piles[i]

    @cache
    def f(i, m):
        if i == n:
            return 0
        if i + 2 * m >= n:
            return suffix[i]
        return max((suffix[i] - f(i + x, max(m, x)) for x in range(1, 2 * m + 1)))
    return f(0, 1)
```

- `suffix=[0]*(n+1)` 加倒序循环：线性求后缀和，suffix[i] = piles[i..n−1] 的总石子数，suffix[n]=0 当哨兵。
- `if i==n: return 0`：没有堆了，当前玩家什么都拿不到。
- `if i+2*m>=n: return suffix[i]`：剩余堆数不超过一次能拿的上限 2M，直接全拿。这个"全拿最优"依赖石子非负——如果有负数堆，多拿可能变亏，这行就不成立了（这正是本章对 Stone II 的前提要求）。
- `return max(suffix[i]-f(i+x,max(m,x)) for x in range(1,2*m+1))`：枚举拿 x 堆；拿完轮到对手，对手在 (i+x, max(M,x)) 能拿 f(...)；剩余总量 suffix[i] 固定，对手拿得越多你剩下的越少，所以你的所得就是 suffix[i] 减对手所得。生成器直接塞进 max，省掉临时列表。
- `@cache`：状态 (i,m) 会大量重复（不同的路径可以到达同一 (i,m)），缓存保证每个状态只算一次。
- `return f(0,1)`：开局位置 0、M=1。

### 4.3 `stone_game_iii(stone_value)`

```python
def stone_game_iii(stone_value):
    n = len(stone_value)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        taken = 0
        dp[i] = float('-inf')
        for j in range(i, min(n, i + 3)):
            taken += stone_value[j]
            dp[i] = max(dp[i], taken - dp[j + 1])
    return 'Alice' if dp[0] > 0 else 'Bob' if dp[0] < 0 else 'Tie'
```

- `dp=[0]*(n+1)`：dp[n]=0 是天然边界（空局面差为 0）。
- `for i in range(n-1,-1,-1)`：从最后一位往前推，保证算 dp[i] 时 dp[j+1] 都已就绪——这是把记忆化递归压成了递推。
- `taken=0; dp[i]=float('-inf')`：taken 边枚举边累加前缀和；初始成负无穷是因为石子可能全为负，任何一个真实选项都比负无穷大，不会漏选。
- `for j in range(i,min(n,i+3))`：j 是"拿到第 j 颗为止"，最多往后拿 3 颗、又不能越过 n。
- `dp[i]=max(dp[i],taken-dp[j+1])`：拿了 [i..j] 这一段（和为 taken），减去对手在剩余 [j+1:] 上的领先差。
- 最后一行把三方结果翻译成字符串：正数 Alice 赢，负数 Bob 赢，0 平局。

## 5. 分段实现：按顺序运行

**本章接口：** `predict_the_winner(nums)`、`stone_game_ii(piles)`、`stone_game_iii(stone_value)`

### 导入本章使用的库

In [1]:
from functools import cache

### predict_the_winner

In [2]:
def predict_the_winner(nums):
    if not nums:
        return True
    dp = list(nums)
    for length in range(2, len(nums) + 1):
        for l in range(len(nums) - length + 1):
            r = l + length - 1
            dp[l] = max(nums[l] - dp[l + 1], nums[r] - dp[l])
    return dp[0] >= 0

### stone_game_ii

In [3]:
def stone_game_ii(piles):
    n = len(piles)
    suffix = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suffix[i] = suffix[i + 1] + piles[i]

    @cache
    def f(i, m):
        if i == n:
            return 0
        if i + 2 * m >= n:
            return suffix[i]
        return max((suffix[i] - f(i + x, max(m, x)) for x in range(1, 2 * m + 1)))
    return f(0, 1)

### stone_game_iii

In [4]:
def stone_game_iii(stone_value):
    n = len(stone_value)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        taken = 0
        dp[i] = float('-inf')
        for j in range(i, min(n, i + 3)):
            taken += stone_value[j]
            dp[i] = max(dp[i], taken - dp[j + 1])
    return 'Alice' if dp[0] > 0 else 'Bob' if dp[0] < 0 else 'Tie'

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,7]; dp=[0]*(len(a)+1); rows=[]
for i in range(len(a)-1,-1,-1):
    options=[sum(a[i:i+k])-dp[i+k] for k in range(1,min(3,len(a)-i)+1)]; dp[i]=max(options); rows.append((i,options,dp[i]))
show_table(['位置','取1..3项得分差','最优差'],rows)

位置,取1..3项得分差,最优差
3,[7],7
2,"[-4, 10]",10
1,"[-8, -2, 12]",12
0,"[-11, -7, -1]",-1


## 7. 正确性、适用条件与复杂度

每个子问题重置为轮到当前行动者，所以下一状态的优势属于对手，需要减去。边界为空时差为0。Stone Game II用剩余总和减去对手最多所得等价于自身最多所得，且所有石子非负时可全取就最优。

**代价：** 两端博弈O(n²)时间O(n)空间；Stone III O(n)时间空间；Stone II保守O(n³)时间O(n²)缓存，递归深度O(n)。II要求非负石堆；III允许负数；平局按接口分别返回True或Tie。

### 展开理解

**为什么"减一下"就实现了双方最优？** 关键在于每个子问题都被重置成"轮到当前行动者"的视角：我走完一步后，棋盘轮到对手，下一个状态算出来的领先差是"对手的领先"。从我的角度看，他领先多少我就落后多少，所以取负号（代码里就是做减法）。又因为总分固定（零和），我最大化自己的差，就同时最小化了对手的差——不需要真的写 min，一个 max 加一个减号就够了。空局面的差是 0，这是递推的起点。

**Stone Game II 为什么能用"总和 − 对手"？** 剩余石子总量 suffix[i] 是定死的，我和对手分的就是这一堆。对手在剩余局面最优能拿 f，那留给我的就是 suffix[i]−f；反过来我在每个选项里挑"剩给对手最少"的那个，等价于挑"自己拿最多"的那个。另外"能全拿就全拿"这一步用了非负性：如果允许负数堆，把负堆留在盘面上可能更划算，这个短路就会给出错误答案——所以本章明确 Stone II 要求非负，Stone III（允许负数）就不做这种短路，老老实实枚举所有选项。

**和概率决策的区别再强调一次：** 概率章里下一状态按概率分布加权求和；本章里对手只有一个选择——对他自己最好（对你最坏）的那个，所以是取 max 而不是求期望。

**复杂度（拿具体数字感受）：** 两端取数是 O(n²) 时间：n=1000 时约 50 万个区间、每个 O(1) 转移，一眨眼算完；空间 O(n)（那一维数组反复覆盖）。Stone III 是 O(n) 时间空间：n=10⁵ 时也就 30 万次基本操作。Stone II 的状态是 (i, M)，M 最多 n 种，每个状态转移最多 2M 步，保守估计 O(n³)：n=100 时约百万级操作，毫无压力；缓存空间 O(n²)，递归深度 O(n)——输入非常长时要防递归栈溢出，可改成迭代。

## 8. 单元测试：每个用例在检查什么

```python
assert not predict_the_winner([1, 5, 2]) and predict_the_winner([1, 5, 233, 7])

assert stone_game_ii([2, 7, 9, 4, 4]) == 10

assert stone_game_iii([1, 2, 3, 7]) == 'Bob' and stone_game_iii([1, 2, 3, 6]) == 'Tie'
```

- 第 1 条：两个官方样例。`[1,5,2]` 先手怎么拿都 3:5 输（第一节手算过）；`[1,5,233,7]` 先手赢——直观理由是奇数位之和 1+233=234 远大于偶数位 5+7=12，先手可以全程锁住 233 那一侧。
- 第 2 条：Stone Game II 官方样例，10 的完整推导见第三节 3.2。
- 第 3 条：Stone Game III 两个样例。`[1,2,3,7]` 的 dp[0]=−1（Bob 赢）；`[1,2,3,6]` 同样手算：dp[3]=6，dp[2]=max(3−6, 9−0)=9，dp[1]=max(2−9, 5−6, 12−0)=12，dp[0]=max(1−12, 3−9, 6−6)=0 → 平局 'Tie'。

后面是三段暴力对拍（都在 “自动测试”部分 里）：

- `ends(a)` 是两端取数的记忆化暴力真值（直接按定义递归，`a[0]-ends(a[1:])` 拿左、`a[-1]-ends(a[:-1])` 拿右）。用它对 `product((-2,0,3),repeat=6)` 生成的全部 729 个六元组（故意包含负数和零）验证 `predict_the_winner(a)==(ends(a)>=0)`。
- `front(a)` 是"取前 1..3 颗"的暴力真值，对同一批 729 个数组验证 `stone_game_iii` 的三向输出（d>0 → Alice，d<0 → Bob，d=0 → Tie）——负数特意被包含，检验"没有全拿短路"的正确性。
- `ii_ref(a,m)` 是 Stone II 的参考递归：直接按"我拿 sum(a[:k])，加上剩余总量减去对手最优"计算。对 `product((1,2),repeat=6)` 的 64 个全非负数组验证 `stone_game_ii(a)==ii_ref(a,1)`。

这三个对拍同时验证了三件事：教学实现与朴素定义一致；含负数时 III 仍对；II 在非负输入下与参考一致。

In [6]:
assert not predict_the_winner([1, 5, 2]) and predict_the_winner([1, 5, 233, 7])

assert stone_game_ii([2, 7, 9, 4, 4]) == 10

assert stone_game_iii([1, 2, 3, 7]) == 'Bob' and stone_game_iii([1, 2, 3, 6]) == 'Tie'

from itertools import product

@cache
def ends(a):
    if not a:
        return 0
    return max(a[0] - ends(a[1:]), a[-1] - ends(a[:-1]))

@cache
def front(a):
    if not a:
        return 0
    return max((sum(a[:k]) - front(a[k:]) for k in range(1, min(3, len(a)) + 1)))

for a in product((-2, 0, 3), repeat=6):
    assert predict_the_winner(a) == (ends(a) >= 0)
    d = front(a)
    assert stone_game_iii(a) == ('Alice' if d > 0 else 'Bob' if d < 0 else 'Tie')

@cache
def ii_ref(a, m):
    if not a:
        return 0
    return max((sum(a[:k]) + (sum(a[k:]) - ii_ref(a[k:], max(m, k))) for k in range(1, min(len(a), 2 * m) + 1)))

for a in product((1, 2), repeat=6):
    assert stone_game_ii(a) == ii_ref(a, 1)

print('N115: 所有本章断言通过')

N115: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导difference=current_gain-next_difference。

**练习 2：** 解释一类题的结论不能替代通用DP。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（推导 difference = current_gain − next_difference）：** 提示——先在纸上写下"状态 = 轮到当前行动者时他能领先多少"，然后想清楚：我做出某个选择后，下一个状态轮到的是对手，那个状态的数值是"对手的领先"，也就是"我的落后"，所以进我的账本时要取负。用 `[1,5,2]` 手算一遍（第 3.1 节有完整过程），再对照 `predict_the_winner` 验证。边界别忘写：空数组差为 0、单元素差为该元素。
- **练习 2（结论不能替代通用 DP）：** 提示——Stone Game（LC877）有个著名结论"偶数堆时先手必胜"，但它的前提（偶数堆、非负）一旦破坏就不成立。你可以用 `predict_the_winner([1,5,2])`（三堆，奇数）当反例：先手反而必输；再看含负数的数组，"先拿大的"这种贪心直觉也会翻车。写清楚你要验证的结论、它的前提，以及一个前提不满足时结论失效的具体数组，最后用本章接口或暴力解跑给你看。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from functools import cache

def predict_the_winner(nums):
    if not nums:
        return True
    dp = list(nums)
    for length in range(2, len(nums) + 1):
        for l in range(len(nums) - length + 1):
            r = l + length - 1
            dp[l] = max(nums[l] - dp[l + 1], nums[r] - dp[l])
    return dp[0] >= 0

def stone_game_ii(piles):
    n = len(piles)
    suffix = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suffix[i] = suffix[i + 1] + piles[i]

    @cache
    def f(i, m):
        if i == n:
            return 0
        if i + 2 * m >= n:
            return suffix[i]
        return max((suffix[i] - f(i + x, max(m, x)) for x in range(1, 2 * m + 1)))
    return f(0, 1)

def stone_game_iii(stone_value):
    n = len(stone_value)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        taken = 0
        dp[i] = float('-inf')
        for j in range(i, min(n, i + 3)):
            taken += stone_value[j]
            dp[i] = max(dp[i], taken - dp[j + 1])
    return 'Alice' if dp[0] > 0 else 'Bob' if dp[0] < 0 else 'Tie'

# 示例与回归测试
assert not predict_the_winner([1, 5, 2]) and predict_the_winner([1, 5, 233, 7])

assert stone_game_ii([2, 7, 9, 4, 4]) == 10

assert stone_game_iii([1, 2, 3, 7]) == 'Bob' and stone_game_iii([1, 2, 3, 6]) == 'Tie'

from itertools import product

@cache
def ends(a):
    if not a:
        return 0
    return max(a[0] - ends(a[1:]), a[-1] - ends(a[:-1]))

@cache
def front(a):
    if not a:
        return 0
    return max((sum(a[:k]) - front(a[k:]) for k in range(1, min(3, len(a)) + 1)))

for a in product((-2, 0, 3), repeat=6):
    assert predict_the_winner(a) == (ends(a) >= 0)
    d = front(a)
    assert stone_game_iii(a) == ('Alice' if d > 0 else 'Bob' if d < 0 else 'Tie')

@cache
def ii_ref(a, m):
    if not a:
        return 0
    return max((sum(a[:k]) + (sum(a[k:]) - ii_ref(a[k:], max(m, k))) for k in range(1, min(len(a), 2 * m) + 1)))

for a in product((1, 2), repeat=6):
    assert stone_game_ii(a) == ii_ref(a, 1)

print('N115: 所有本章断言通过')

N115: 所有本章断言通过


## 11. 代表题与迁移

- **486. Predict the Winner**：这就是 `predict_the_winner` 的原题——练"区间 + 得分差 + 一维滚动"的基本功。
- **877. Stone Game**：与 486 同一个模型（两端取、比总分），练的是把"先手必胜"的巧结论和通用 DP 互相印证。
- **1140. Stone Game II**：这就是 `stone_game_ii` 的原题——练"把会变的参数 M 放进状态"以及"我的所得 = 剩余总和 − 对手所得"的转化。
- **1406. Stone Game III**：这就是 `stone_game_iii` 的原题——练"取前 1..3 的前缀 + 允许负值 + 三向输出（Alice/Bob/Tie）"。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。